# Smart Retail Analytics - Data Mining Lab Project

**Dataset:** UCI Online Retail

**Name:** Muhammad Awais Ashraf </br>
**Reg No:** FA23-BDS-053  </br>
**Class:** BDS-5

**Name:** [Mohsin Nadeem] </br>
**Reg No:** FA23-BDS-021 </br>
**Class:** BDS-5


## Project Overview
This project looks at real sales data from a store to understand how customers shop. Using Python, it finds shopping habits and builds models to predict customer behavior. The data includes things like order number, product code, product name, amount bought, date, price, customer ID, and country.

## Table of Contents
1. [Introduction & Preprocessing](#1-data-ingestion--preprocessing)
2. [Similarity & Dissimilarity Matrices](#2-similarity--dissimilarity-matrices)
3. [Clustering Analysis](#3-clustering-analysis)
4. [Association Rules](#4-association-rule-mining)
5. [Naïve Bayes Classification](#5-naïve-bayes-classification)
6. [Support Vector Machine (SVM)](#6-support-vector-machine)
7. [Conclusions & Business Recommendations](#7-conclusions--business-recommendations)

## 1. Introduction & Preprocessing

In [3]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('default')
sns.set_palette("husl")

print("Libraries imported successfully!")

ModuleNotFoundError: No module named 'matplotlib'

In [2]:
# Load the dataset
df = pd.read_excel('Online Retail.xlsx')

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
df.head()

NameError: name 'pd' is not defined

In [94]:
# Data exploration
print("Data Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nBasic Statistics:")
df.describe()

In [95]:
# Data cleaning steps
print(f"Original dataset size: {len(df)}")

# Remove rows with missing CustomerID
df_clean = df.dropna(subset=['CustomerID']).copy()
print(f"After removing missing CustomerIDs: {len(df_clean)}")

# Remove cancellations (InvoiceNo starting with 'C')
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]
print(f"After removing cancellations: {len(df_clean)}")

# Remove negative quantities and zero prices
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]
print(f"After removing negative quantities and zero prices: {len(df_clean)}")

# Convert CustomerID to integer
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)

print("\nCleaning completed!")

In [96]:
# Feature Engineering
# Create TotalAmount feature
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']

print(f"TotalAmount statistics:")
print(df_clean['TotalAmount'].describe())

# Check for any extreme outliers
print(f"\nTop 10 highest transaction amounts:")
print(df_clean.nlargest(10, 'TotalAmount')[['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'UnitPrice', 'TotalAmount', 'CustomerID']])

In [97]:
# RFM Analysis (Recency, Frequency, Monetary)
# Calculate reference date (latest date in dataset + 1 day)
reference_date = df_clean['InvoiceDate'].max() + timedelta(days=1)
print(f"Reference date for recency calculation: {reference_date}")

# Calculate RFM metrics per customer
rfm = df_clean.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',  # Frequency
    'TotalAmount': 'sum'  # Monetary
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

print(f"RFM dataset shape: {rfm.shape}")
print(f"\nRFM statistics:")
print(rfm.describe())

# Display first few rows
print(f"\nFirst 10 customers RFM data:")
rfm.head(10)

In [98]:
# One-hot encode Country feature
from sklearn.preprocessing import LabelEncoder

# First, let's see the distribution of countries
print("Country distribution:")
print(df_clean['Country'].value_counts().head(10))

# Create country features per customer (most frequent country)
customer_country = df_clean.groupby('CustomerID')['Country'].agg(lambda x: x.mode()[0]).reset_index()

# One-hot encode countries (keeping only top countries to avoid too many features)
top_countries = df_clean['Country'].value_counts().head(10).index.tolist()
customer_country['Country_encoded'] = customer_country['Country'].apply(
    lambda x: x if x in top_countries else 'Other'
)

# Create dummy variables
country_dummies = pd.get_dummies(customer_country['Country_encoded'], prefix='Country')
customer_features = pd.concat([customer_country[['CustomerID']], country_dummies], axis=1)

print(f"\nCountry encoding completed. Shape: {customer_features.shape}")
print(f"Encoded countries: {country_dummies.columns.tolist()}")

In [99]:
# Merge RFM with country features
rfm_final = rfm.merge(customer_features, on='CustomerID', how='left')

print(f"Final RFM dataset shape: {rfm_final.shape}")
print(f"\nFinal dataset columns: {rfm_final.columns.tolist()}")
print(f"\nSample of final preprocessed data:")
rfm_final.head()

In [100]:
# Visualize RFM distributions
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('RFM Analysis - Customer Behavior Distribution', fontsize=16)

# Recency distribution
axes[0,0].hist(rfm['Recency'], bins=50, alpha=0.7, color='skyblue')
axes[0,0].set_title('Recency Distribution (Days since last purchase)')
axes[0,0].set_xlabel('Days')
axes[0,0].set_ylabel('Frequency')

# Frequency distribution
axes[0,1].hist(rfm['Frequency'], bins=50, alpha=0.7, color='lightgreen')
axes[0,1].set_title('Frequency Distribution (Number of orders)')
axes[0,1].set_xlabel('Number of Orders')
axes[0,1].set_ylabel('Frequency')

# Monetary distribution (log scale due to skewness)
axes[1,0].hist(np.log1p(rfm['Monetary']), bins=50, alpha=0.7, color='salmon')
axes[1,0].set_title('Monetary Distribution (Log scale)')
axes[1,0].set_xlabel('Log(Total Amount + 1)')
axes[1,0].set_ylabel('Frequency')

# RFM correlation heatmap
rfm_corr = rfm[['Recency', 'Frequency', 'Monetary']].corr()
sns.heatmap(rfm_corr, annot=True, cmap='coolwarm', center=0, ax=axes[1,1])
axes[1,1].set_title('RFM Features Correlation')

plt.tight_layout()
plt.show()

print("✅ Data Preprocessing Completed!")
print(f"📊 Ready for analysis with {len(rfm_final)} customers")

## 2. Similarity & Dissimilarity Matrices

In this section, we'll compute:
- Euclidean distance between customers in RFM space
- Jaccard similarity based on top 10 most purchased products per customer

In [101]:
# Import required libraries for similarity analysis
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances, pairwise_distances
from scipy.spatial.distance import pdist, squareform
import itertools

# Prepare RFM data for distance calculation
rfm_features = rfm[['Recency', 'Frequency', 'Monetary']].copy()

# Standardize RFM features for fair distance calculation
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_features)
rfm_scaled_df = pd.DataFrame(rfm_scaled, columns=['Recency_scaled', 'Frequency_scaled', 'Monetary_scaled'])
rfm_scaled_df['CustomerID'] = rfm['CustomerID'].values

print(f"Scaled RFM data shape: {rfm_scaled_df.shape}")
print("\nScaled RFM statistics:")
print(rfm_scaled_df.describe())

In [102]:
# Calculate Euclidean distance matrix (using subset of customers for visualization)
# Using first 50 customers for computational efficiency and visualization clarity
subset_size = 50
rfm_subset = rfm_scaled[:subset_size]
customer_ids_subset = rfm['CustomerID'].iloc[:subset_size].values

# Compute Euclidean distance matrix
euclidean_dist_matrix = euclidean_distances(rfm_subset)

print(f"Euclidean distance matrix shape: {euclidean_dist_matrix.shape}")
print(f"Sample distances between first 5 customers:")
print(euclidean_dist_matrix[:5, :5])

# Visualize Euclidean distance matrix as heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(euclidean_dist_matrix, 
            xticklabels=customer_ids_subset,
            yticklabels=customer_ids_subset,
            cmap='viridis',
            cbar_kws={'label': 'Euclidean Distance'})
plt.title(f'Euclidean Distance Matrix - RFM Space\n(First {subset_size} Customers)', fontsize=14)
plt.xlabel('Customer ID')
plt.ylabel('Customer ID')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [103]:
# Prepare data for Jaccard similarity (top 10 products per customer)
# Get top purchased products per customer
customer_products = df_clean.groupby(['CustomerID', 'StockCode'])['Quantity'].sum().reset_index()

# Get top 10 products for each customer
top_products_per_customer = customer_products.groupby('CustomerID').apply(
    lambda x: x.nlargest(10, 'Quantity')['StockCode'].tolist()
).reset_index()
top_products_per_customer.columns = ['CustomerID', 'TopProducts']

print(f"Number of customers with product data: {len(top_products_per_customer)}")
print("\nSample top products for first 5 customers:")
for i in range(5):
    customer_id = top_products_per_customer.iloc[i]['CustomerID']
    products = top_products_per_customer.iloc[i]['TopProducts']
    print(f"Customer {customer_id}: {products}")

In [104]:
# Calculate Jaccard similarity matrix
def jaccard_similarity(set1, set2):
    """Calculate Jaccard similarity between two sets"""
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union > 0 else 0

# Use subset for visualization
top_products_subset = top_products_per_customer.head(subset_size)
n_customers = len(top_products_subset)

# Initialize Jaccard similarity matrix
jaccard_matrix = np.zeros((n_customers, n_customers))

# Calculate pairwise Jaccard similarities
for i in range(n_customers):
    for j in range(n_customers):
        if i == j:
            jaccard_matrix[i, j] = 1.0  # Self-similarity
        else:
            set1 = set(top_products_subset.iloc[i]['TopProducts'])
            set2 = set(top_products_subset.iloc[j]['TopProducts'])
            jaccard_matrix[i, j] = jaccard_similarity(set1, set2)

print(f"Jaccard similarity matrix shape: {jaccard_matrix.shape}")
print(f"Sample Jaccard similarities between first 5 customers:")
print(jaccard_matrix[:5, :5])

# Visualize Jaccard similarity matrix
plt.figure(figsize=(12, 10))
customer_ids_subset_jaccard = top_products_subset['CustomerID'].values
sns.heatmap(jaccard_matrix,
            xticklabels=customer_ids_subset_jaccard,
            yticklabels=customer_ids_subset_jaccard,
            cmap='Blues',
            vmin=0, vmax=1,
            cbar_kws={'label': 'Jaccard Similarity'})
plt.title(f'Jaccard Similarity Matrix - Product Purchase Patterns\n(First {subset_size} Customers)', fontsize=14)
plt.xlabel('Customer ID')
plt.ylabel('Customer ID')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [105]:
# Analysis of similarity matrices
print("📊 SIMILARITY ANALYSIS SUMMARY")
print("=" * 40)

# Euclidean distance analysis
print("\n🔹 Euclidean Distance Analysis (RFM Space):")
non_diag_euclidean = euclidean_dist_matrix[np.triu_indices_from(euclidean_dist_matrix, k=1)]
print(f"   • Average distance between customers: {non_diag_euclidean.mean():.3f}")
print(f"   • Min distance: {non_diag_euclidean.min():.3f}")
print(f"   • Max distance: {non_diag_euclidean.max():.3f}")
print(f"   • Standard deviation: {non_diag_euclidean.std():.3f}")

# Jaccard similarity analysis
print("\n🔹 Jaccard Similarity Analysis (Product Overlap):")
non_diag_jaccard = jaccard_matrix[np.triu_indices_from(jaccard_matrix, k=1)]
print(f"   • Average similarity: {non_diag_jaccard.mean():.3f}")
print(f"   • Min similarity: {non_diag_jaccard.min():.3f}")
print(f"   • Max similarity: {non_diag_jaccard.max():.3f}")
print(f"   • Standard deviation: {non_diag_jaccard.std():.3f}")

# Find most similar and dissimilar customer pairs
max_jaccard_idx = np.unravel_index(np.argmax(jaccard_matrix - np.eye(n_customers)), jaccard_matrix.shape)
min_euclidean_idx = np.unravel_index(np.argmin(euclidean_dist_matrix + np.eye(subset_size) * 1000), euclidean_dist_matrix.shape)

print(f"\n🔹 Most Similar Customers:")
print(f"   • By product overlap: Customers {customer_ids_subset_jaccard[max_jaccard_idx[0]]} & {customer_ids_subset_jaccard[max_jaccard_idx[1]]} (Jaccard: {jaccard_matrix[max_jaccard_idx]:.3f})")
print(f"   • By RFM behavior: Customers {customer_ids_subset[min_euclidean_idx[0]]} & {customer_ids_subset[min_euclidean_idx[1]]} (Distance: {euclidean_dist_matrix[min_euclidean_idx]:.3f})")

print("\n✅ Similarity & Dissimilarity Analysis Completed!")

## 3. Clustering Analysis

We'll perform customer segmentation using:
- K-Means clustering with optimal K selection
- DBSCAN clustering with parameter tuning

In [106]:
# Import clustering libraries
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples
import matplotlib.patches as patches

# Prepare data for clustering (using scaled RFM features)
X_clustering = rfm_scaled

print(f"Data for clustering shape: {X_clustering.shape}")
print("Using standardized RFM features for clustering...")

In [107]:
# K-Means: Find optimal number of clusters using Elbow Method
K_range = range(2, 11)
inertias = []
silhouette_scores = []

print("Finding optimal K for K-Means clustering...")
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_clustering)
    
    inertias.append(kmeans.inertia_)
    sil_score = silhouette_score(X_clustering, cluster_labels)
    silhouette_scores.append(sil_score)
    
    print(f"K={k}: Inertia={kmeans.inertia_:.2f}, Silhouette Score={sil_score:.3f}")

# Plot Elbow Curve and Silhouette Scores
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Elbow plot
ax1.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Inertia (Within-cluster sum of squares)')
ax1.set_title('Elbow Method for Optimal K')
ax1.grid(True, alpha=0.3)

# Silhouette score plot
ax2.plot(K_range, silhouette_scores, 'ro-', linewidth=2, markersize=8)
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score vs Number of Clusters')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Select optimal K (highest silhouette score)
optimal_k = K_range[np.argmax(silhouette_scores)]
print(f"\n🏆 Optimal K selected: {optimal_k} (Silhouette Score: {max(silhouette_scores):.3f})")

In [108]:
# Apply K-Means with optimal K
kmeans_optimal = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
kmeans_labels = kmeans_optimal.fit_predict(X_clustering)

# Add cluster labels to RFM data
rfm_clustered = rfm.copy()
rfm_clustered['KMeans_Cluster'] = kmeans_labels

print(f"K-Means clustering completed with K={optimal_k}")
print(f"\nCluster distribution:")
print(rfm_clustered['KMeans_Cluster'].value_counts().sort_index())

# Analyze cluster characteristics
print(f"\n📊 K-Means Cluster Characteristics:")
cluster_summary = rfm_clustered.groupby('KMeans_Cluster')[['Recency', 'Frequency', 'Monetary']].agg(['mean', 'std'])
print(cluster_summary.round(2))

In [109]:
# Visualize K-Means clusters using PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_clustering)

# Create scatter plot
plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_labels, 
                    cmap='viridis', alpha=0.7, s=50)
plt.colorbar(scatter, label='Cluster')

# Plot cluster centers in PCA space
centers_pca = pca.transform(kmeans_optimal.cluster_centers_)
plt.scatter(centers_pca[:, 0], centers_pca[:, 1], 
           c='red', marker='x', s=200, linewidths=3, label='Centroids')

plt.xlabel(f'First Principal Component (Explained Variance: {pca.explained_variance_ratio_[0]:.2%})')
plt.ylabel(f'Second Principal Component (Explained Variance: {pca.explained_variance_ratio_[1]:.2%})')
plt.title(f'K-Means Clustering Visualization (K={optimal_k})\nPCA Projection of RFM Space')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"PCA Explained Variance Ratio: {pca.explained_variance_ratio_}")
print(f"Total Explained Variance: {pca.explained_variance_ratio_.sum():.2%}")

In [110]:
# DBSCAN Clustering
print("\n" + "="*50)
print("🔄 DBSCAN CLUSTERING")
print("="*50)

# Parameter tuning for DBSCAN
from sklearn.neighbors import NearestNeighbors

# Find optimal eps using k-distance plot
k = 4  # MinPts = k + 1 = 5
neighbors = NearestNeighbors(n_neighbors=k)
neighbors_fit = neighbors.fit(X_clustering)
distances, indices = neighbors_fit.kneighbors(X_clustering)
distances = np.sort(distances[:, k-1], axis=0)

plt.figure(figsize=(10, 6))
plt.plot(range(len(distances)), distances)
plt.xlabel('Points sorted by distance')
plt.ylabel(f'{k}-NN Distance')
plt.title('K-Distance Plot for DBSCAN eps Parameter Selection')
plt.grid(True, alpha=0.3)
plt.show()

# Test different eps and min_samples values
eps_values = [0.5, 0.8, 1.0, 1.2, 1.5]
min_samples_values = [3, 5, 7, 10]

dbscan_results = []

print("Testing DBSCAN parameters...")
for eps in eps_values:
    for min_samples in min_samples_values:
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        labels = dbscan.fit_predict(X_clustering)
        
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1)
        
        if n_clusters > 1:  # Only calculate silhouette for valid clusterings
            sil_score = silhouette_score(X_clustering, labels) if n_clusters > 1 and n_noise < len(labels) else -1
        else:
            sil_score = -1
            
        dbscan_results.append({
            'eps': eps,
            'min_samples': min_samples,
            'n_clusters': n_clusters,
            'n_noise': n_noise,
            'silhouette': sil_score
        })
        
        print(f"eps={eps}, min_samples={min_samples}: {n_clusters} clusters, {n_noise} noise points, silhouette={sil_score:.3f}")

# Select best DBSCAN parameters
valid_results = [r for r in dbscan_results if r['silhouette'] > 0 and r['n_clusters'] > 1]
if valid_results:
    best_dbscan = max(valid_results, key=lambda x: x['silhouette'])
    print(f"\n🏆 Best DBSCAN parameters: eps={best_dbscan['eps']}, min_samples={best_dbscan['min_samples']}")
    print(f"   Silhouette Score: {best_dbscan['silhouette']:.3f}")
else:
    # Fallback parameters
    best_dbscan = {'eps': 1.0, 'min_samples': 5}
    print(f"\n⚠️ Using fallback DBSCAN parameters: eps={best_dbscan['eps']}, min_samples={best_dbscan['min_samples']}")

In [111]:
# Apply DBSCAN with best parameters
dbscan_optimal = DBSCAN(eps=best_dbscan['eps'], min_samples=best_dbscan['min_samples'])
dbscan_labels = dbscan_optimal.fit_predict(X_clustering)

# Add DBSCAN labels to RFM data
rfm_clustered['DBSCAN_Cluster'] = dbscan_labels

# Analyze DBSCAN results
n_clusters_dbscan = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)

print(f"\nDBSCAN Results:")
print(f"   • Number of clusters: {n_clusters_dbscan}")
print(f"   • Number of noise points: {n_noise}")
print(f"   • Percentage of noise: {n_noise/len(dbscan_labels)*100:.1f}%")

if n_clusters_dbscan > 0:
    print(f"\nDBSCAN Cluster distribution:")
    unique, counts = np.unique(dbscan_labels, return_counts=True)
    for cluster, count in zip(unique, counts):
        if cluster == -1:
            print(f"   Noise points: {count}")
        else:
            print(f"   Cluster {cluster}: {count} points")

# Visualize DBSCAN clusters
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# K-Means visualization
scatter1 = ax1.scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_labels, 
                      cmap='viridis', alpha=0.7, s=50)
ax1.scatter(centers_pca[:, 0], centers_pca[:, 1], 
           c='red', marker='x', s=200, linewidths=3)
ax1.set_title(f'K-Means Clustering (K={optimal_k})')
ax1.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax1.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax1.grid(True, alpha=0.3)

# DBSCAN visualization
scatter2 = ax2.scatter(X_pca[:, 0], X_pca[:, 1], c=dbscan_labels, 
                      cmap='viridis', alpha=0.7, s=50)
ax2.set_title(f'DBSCAN Clustering (eps={best_dbscan["eps"]}, min_samples={best_dbscan["min_samples"]})')
ax2.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax2.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ Clustering Analysis Completed!")

In [112]:
# Compare clustering results
print("📊 CLUSTERING COMPARISON")
print("=" * 40)

# K-Means analysis
if optimal_k > 1:
    kmeans_silhouette = silhouette_score(X_clustering, kmeans_labels)
    print(f"\n🔹 K-Means (K={optimal_k}):")
    print(f"   • Silhouette Score: {kmeans_silhouette:.3f}")
    print(f"   • All points clustered (no noise)")
    
    # Cluster interpretation
    print(f"\n   📊 Cluster Insights:")
    for cluster in sorted(rfm_clustered['KMeans_Cluster'].unique()):
        cluster_data = rfm_clustered[rfm_clustered['KMeans_Cluster'] == cluster]
        avg_recency = cluster_data['Recency'].mean()
        avg_frequency = cluster_data['Frequency'].mean()
        avg_monetary = cluster_data['Monetary'].mean()
        
        # Classify cluster type
        if avg_recency < rfm['Recency'].median() and avg_frequency > rfm['Frequency'].median():
            cluster_type = "Active Customers"
        elif avg_monetary > rfm['Monetary'].quantile(0.75):
            cluster_type = "High Value Customers"
        elif avg_recency > rfm['Recency'].quantile(0.75):
            cluster_type = "At-Risk/Dormant Customers"
        else:
            cluster_type = "Regular Customers"
            
        print(f"      Cluster {cluster} ({len(cluster_data)} customers): {cluster_type}")
        print(f"         R={avg_recency:.0f} days, F={avg_frequency:.1f} orders, M=${avg_monetary:.0f}")

# DBSCAN analysis
if n_clusters_dbscan > 1:
    # Calculate silhouette only for non-noise points
    non_noise_mask = dbscan_labels != -1
    if np.sum(non_noise_mask) > 1:
        dbscan_silhouette = silhouette_score(X_clustering[non_noise_mask], dbscan_labels[non_noise_mask])
        print(f"\n🔹 DBSCAN (eps={best_dbscan['eps']}, min_samples={best_dbscan['min_samples']}):")
        print(f"   • Silhouette Score: {dbscan_silhouette:.3f} (excluding noise)")
        print(f"   • Identified {n_noise} outlier customers ({n_noise/len(dbscan_labels)*100:.1f}% noise)")
        print(f"   • Found {n_clusters_dbscan} natural clusters")
else:
    print(f"\n🔹 DBSCAN: Unable to find meaningful clusters with tested parameters")

print(f"\n📁 Final clustered dataset shape: {rfm_clustered.shape}")
print(f"Columns: {rfm_clustered.columns.tolist()}")

## 4. Association Rule Mining

We'll discover frequent itemsets and association rules using:
- Apriori algorithm
- FP-Growth algorithm

Both will be applied to market basket analysis to find product associations.

In [113]:
# Import association rule mining libraries
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder
import time

# Prepare market basket data
print("Preparing market basket data for association rule mining...")

# Create transaction data (each invoice as a transaction)
transactions = df_clean.copy()
transactions['StockCode'] = transactions['StockCode'].astype(str)
transactions = transactions.groupby('InvoiceNo')['StockCode'].apply(list).reset_index()
transactions.columns = ['InvoiceNo', 'Items']

print(f"Total transactions: {len(transactions)}")
print(f"Sample transactions:")
for i in range(3):
    print(f"  Invoice {transactions.iloc[i]['InvoiceNo']}: {transactions.iloc[i]['Items'][:10]}...")  # Show first 10 items

# Get transaction lists
transaction_list = transactions['Items'].tolist()

# Remove transactions with only 1 item (can't form associations)
transaction_list = [transaction for transaction in transaction_list if len(transaction) > 1]
print(f"Transactions with 2+ items: {len(transaction_list)}")

In [114]:
# Encode transactions for association rule mining
te = TransactionEncoder()
te_ary = te.fit(transaction_list).transform(transaction_list)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

print(f"Encoded transaction matrix shape: {df_encoded.shape}")
print(f"Number of unique items: {len(te.columns_)}")

# Analyze item frequencies
item_support = df_encoded.mean().sort_values(ascending=False)
print(f"\nTop 20 most frequent items:")
print(item_support.head(20))

# Filter items with minimum support for computational efficiency
min_item_support = 0.01  # Items appearing in at least 1% of transactions
frequent_items = item_support[item_support >= min_item_support].index.tolist()
df_filtered = df_encoded[frequent_items]

print(f"\nItems with support >= {min_item_support}: {len(frequent_items)}")
print(f"Filtered transaction matrix shape: {df_filtered.shape}")

# Visualize top items
plt.figure(figsize=(12, 6))
top_20_items = item_support.head(20)
plt.bar(range(len(top_20_items)), top_20_items.values, alpha=0.7)
plt.xlabel('Items (ranked by frequency)')
plt.ylabel('Support (proportion of transactions)')
plt.title('Top 20 Most Frequent Items in Transactions')
plt.xticks(range(len(top_20_items)), [str(item)[:10] + '...' if len(str(item)) > 10 else str(item) for item in top_20_items.index], rotation=45)
plt.tight_layout()
plt.show()

In [115]:
# Apply Apriori Algorithm
print("\n" + "="*50)
print("🔍 APRIORI ALGORITHM")
print("="*50)

min_support = 0.02  # 2% minimum support
min_confidence = 0.6  # 60% minimum confidence
min_lift = 1.2  # 120% minimum lift

print(f"Parameters: min_support={min_support}, min_confidence={min_confidence}, min_lift={min_lift}")

# Measure execution time
start_time = time.time()

# Find frequent itemsets using Apriori
frequent_itemsets_apriori = apriori(df_filtered, min_support=min_support, use_colnames=True)
apriori_time = time.time() - start_time

print(f"\nApriori execution time: {apriori_time:.2f} seconds")
print(f"Frequent itemsets found: {len(frequent_itemsets_apriori)}")

if len(frequent_itemsets_apriori) > 0:
    print(f"\nTop 10 frequent itemsets by support:")
    print(frequent_itemsets_apriori.nlargest(10, 'support'))
    
    # Generate association rules
    rules_apriori = association_rules(frequent_itemsets_apriori, 
                                    metric="confidence", 
                                    min_threshold=min_confidence)
    
    # Filter by lift
    rules_apriori = rules_apriori[rules_apriori['lift'] >= min_lift]
    
    print(f"\nAssociation rules generated: {len(rules_apriori)}")
    
    if len(rules_apriori) > 0:
        # Sort by lift and show top rules
        rules_apriori_sorted = rules_apriori.sort_values('lift', ascending=False)
        print(f"\nTop 10 association rules by lift:")
        display_cols = ['antecedents', 'consequents', 'support', 'confidence', 'lift']
        print(rules_apriori_sorted[display_cols].head(10).to_string(index=False))
else:
    print("No frequent itemsets found with current parameters.")
    rules_apriori = pd.DataFrame()

In [116]:
# Apply FP-Growth Algorithm
print("\n" + "="*50)
print("🌳 FP-GROWTH ALGORITHM")
print("="*50)

# Measure execution time for FP-Growth
start_time = time.time()

# Find frequent itemsets using FP-Growth
frequent_itemsets_fpgrowth = fpgrowth(df_filtered, min_support=min_support, use_colnames=True)
fpgrowth_time = time.time() - start_time

print(f"FP-Growth execution time: {fpgrowth_time:.2f} seconds")
print(f"Frequent itemsets found: {len(frequent_itemsets_fpgrowth)}")

if len(frequent_itemsets_fpgrowth) > 0:
    print(f"\nTop 10 frequent itemsets by support:")
    print(frequent_itemsets_fpgrowth.nlargest(10, 'support'))
    
    # Generate association rules
    rules_fpgrowth = association_rules(frequent_itemsets_fpgrowth, 
                                     metric="confidence", 
                                     min_threshold=min_confidence)
    
    # Filter by lift
    rules_fpgrowth = rules_fpgrowth[rules_fpgrowth['lift'] >= min_lift]
    
    print(f"\nAssociation rules generated: {len(rules_fpgrowth)}")
    
    if len(rules_fpgrowth) > 0:
        # Sort by lift and show top rules
        rules_fpgrowth_sorted = rules_fpgrowth.sort_values('lift', ascending=False)
        print(f"\nTop 10 association rules by lift:")
        display_cols = ['antecedents', 'consequents', 'support', 'confidence', 'lift']
        print(rules_fpgrowth_sorted[display_cols].head(10).to_string(index=False))
else:
    print("No frequent itemsets found with current parameters.")
    rules_fpgrowth = pd.DataFrame()

In [117]:
# Compare Apriori vs FP-Growth
print("\n" + "="*50)
print("📊 ASSOCIATION RULE MINING COMPARISON")
print("="*50)

# Runtime comparison
print(f"\n🔹 Runtime Comparison:")
print(f"   • Apriori: {apriori_time:.2f} seconds")
print(f"   • FP-Growth: {fpgrowth_time:.2f} seconds")
print(f"   • Speedup: {apriori_time/fpgrowth_time:.2f}x" if fpgrowth_time > 0 else "   • FP-Growth was much faster")

# Results comparison
print(f"\n🔹 Results Comparison:")
print(f"   • Apriori frequent itemsets: {len(frequent_itemsets_apriori)}")
print(f"   • FP-Growth frequent itemsets: {len(frequent_itemsets_fpgrowth)}")
print(f"   • Apriori association rules: {len(rules_apriori) if 'rules_apriori' in locals() else 0}")
print(f"   • FP-Growth association rules: {len(rules_fpgrowth) if 'rules_fpgrowth' in locals() else 0}")

# Create summary table of top rules
if len(rules_fpgrowth) > 0:  # Use FP-Growth results (typically same as Apriori but faster)
    top_rules = rules_fpgrowth_sorted.head(10).copy()
    
    # Create readable rule descriptions
    top_rules['Rule'] = top_rules.apply(
        lambda row: f"{list(row['antecedents'])[0]} → {list(row['consequents'])[0]}", axis=1
    )
    
    print(f"\n📈 TOP 10 ASSOCIATION RULES:")
    print("-" * 80)
    for idx, rule in top_rules.iterrows():
        print(f"{rule['Rule']:50} | Conf: {rule['confidence']:.3f} | Lift: {rule['lift']:.3f}")
    
    # Visualize rules
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Support vs Confidence scatter
    scatter1 = ax1.scatter(top_rules['support'], top_rules['confidence'], 
                          c=top_rules['lift'], s=100, alpha=0.7, cmap='viridis')
    ax1.set_xlabel('Support')
    ax1.set_ylabel('Confidence')
    ax1.set_title('Association Rules: Support vs Confidence\n(Color = Lift)')
    plt.colorbar(scatter1, ax=ax1, label='Lift')
    ax1.grid(True, alpha=0.3)
    
    # Lift distribution
    ax2.hist(rules_fpgrowth['lift'], bins=20, alpha=0.7, color='skyblue', edgecolor='black')
    ax2.axvline(rules_fpgrowth['lift'].mean(), color='red', linestyle='--', 
               label=f'Mean Lift: {rules_fpgrowth["lift"].mean():.2f}')
    ax2.set_xlabel('Lift')
    ax2.set_ylabel('Frequency')
    ax2.set_title('Distribution of Lift Values')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
else:
    print("\n⚠️ No association rules found with current parameters.")
    print("Consider lowering min_support, min_confidence, or min_lift thresholds.")

print("\n✅ Association Rule Mining Completed!")

## 5. Naïve Bayes Classification

We'll build classification models to predict "High Value" customers using:
- GaussianNB on RFM features
- BernoulliNB on binarized purchase flags

**Target Variable:** High Value = Monetary > 75th percentile

In [118]:
# Import classification libraries
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Create target variable: High Value customers (Monetary > 75th percentile)
monetary_75th = rfm['Monetary'].quantile(0.75)
rfm_classification = rfm.copy()
rfm_classification['HighValue'] = (rfm_classification['Monetary'] > monetary_75th).astype(int)

print(f"Monetary 75th percentile threshold: ${monetary_75th:.2f}")
print(f"\nTarget variable distribution:")
print(rfm_classification['HighValue'].value_counts())
print(f"\nPercentage of high-value customers: {rfm_classification['HighValue'].mean()*100:.1f}%")

# Display sample data
print(f"\nSample classification data:")
print(rfm_classification[['CustomerID', 'Recency', 'Frequency', 'Monetary', 'HighValue']].head(10))

In [119]:
# GaussianNB on RFM features
print("\n" + "="*50)
print("🧠 GAUSSIAN NAIVE BAYES (RFM Features)")
print("="*50)

# Prepare features and target
X_rfm = rfm_classification[['Recency', 'Frequency', 'Monetary']]
y = rfm_classification['HighValue']

# Split data
X_train_rfm, X_test_rfm, y_train, y_test = train_test_split(
    X_rfm, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training set size: {len(X_train_rfm)}")
print(f"Test set size: {len(X_test_rfm)}")
print(f"Training set class distribution: {y_train.value_counts().to_dict()}")

# Scale features for better performance
scaler_rfm = StandardScaler()
X_train_rfm_scaled = scaler_rfm.fit_transform(X_train_rfm)
X_test_rfm_scaled = scaler_rfm.transform(X_test_rfm)

# Train Gaussian Naive Bayes
gnb = GaussianNB()
gnb.fit(X_train_rfm_scaled, y_train)

# Predictions
y_pred_gnb = gnb.predict(X_test_rfm_scaled)
y_prob_gnb = gnb.predict_proba(X_test_rfm_scaled)[:, 1]

# Evaluate model
print(f"\n📊 Gaussian Naive Bayes Results:")
print(classification_report(y_test, y_pred_gnb, target_names=['Regular', 'High Value']))

# Confusion Matrix
cm_gnb = confusion_matrix(y_test, y_pred_gnb)
print(f"\nConfusion Matrix:")
print(cm_gnb)

In [120]:
# Prepare binarized purchase flags for BernoulliNB
print("\n" + "="*50)
print("🔢 PREPARING BINARIZED PURCHASE FLAGS")
print("="*50)

# Get top products for creating purchase flags
top_n_products = 50  # Use top 50 products
top_products = df_clean['StockCode'].value_counts().head(top_n_products).index.tolist()

print(f"Using top {top_n_products} products for purchase flags")
print(f"Top 10 products: {top_products[:10]}")

# Create purchase flags per customer
customer_purchase_flags = pd.DataFrame()
customer_purchase_flags['CustomerID'] = rfm['CustomerID']

# For each top product, create a binary flag indicating if customer purchased it
for product in top_products:
    customers_who_bought = df_clean[df_clean['StockCode'] == product]['CustomerID'].unique()
    customer_purchase_flags[f'Bought_{product}'] = customer_purchase_flags['CustomerID'].isin(customers_who_bought).astype(int)

print(f"\nPurchase flags dataset shape: {customer_purchase_flags.shape}")
print(f"Purchase flag statistics (first 5 products):")
for col in customer_purchase_flags.columns[1:6]:  # First 5 product columns
    print(f"   {col}: {customer_purchase_flags[col].sum()} customers ({customer_purchase_flags[col].mean()*100:.1f}%)")

In [121]:
# BernoulliNB on binarized purchase flags
print("\n" + "="*50)
print("🔢 BERNOULLI NAIVE BAYES (Purchase Flags)")
print("="*50)

# Merge purchase flags with target variable
purchase_data = customer_purchase_flags.merge(
    rfm_classification[['CustomerID', 'HighValue']], 
    on='CustomerID', how='inner'
)

# Prepare features (excluding CustomerID)
X_purchase = purchase_data.drop(['CustomerID', 'HighValue'], axis=1)
y_purchase = purchase_data['HighValue']

print(f"Purchase flags feature matrix shape: {X_purchase.shape}")
print(f"Number of features (products): {X_purchase.shape[1]}")

# Split data
X_train_purchase, X_test_purchase, y_train_purchase, y_test_purchase = train_test_split(
    X_purchase, y_purchase, test_size=0.3, random_state=42, stratify=y_purchase
)

# Train Bernoulli Naive Bayes
bnb = BernoulliNB()
bnb.fit(X_train_purchase, y_train_purchase)

# Predictions
y_pred_bnb = bnb.predict(X_test_purchase)
y_prob_bnb = bnb.predict_proba(X_test_purchase)[:, 1]

# Evaluate model
print(f"\n📊 Bernoulli Naive Bayes Results:")
print(classification_report(y_test_purchase, y_pred_bnb, target_names=['Regular', 'High Value']))

# Confusion Matrix
cm_bnb = confusion_matrix(y_test_purchase, y_pred_bnb)
print(f"\nConfusion Matrix:")
print(cm_bnb)

In [122]:
# Visualize Naive Bayes results
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Naïve Bayes Classification Results', fontsize=16)

# Confusion matrices
from sklearn.metrics import ConfusionMatrixDisplay

# GaussianNB confusion matrix
disp1 = ConfusionMatrixDisplay(confusion_matrix=cm_gnb, display_labels=['Regular', 'High Value'])
disp1.plot(ax=axes[0,0], cmap='Blues')
axes[0,0].set_title('Gaussian NB - Confusion Matrix')

# BernoulliNB confusion matrix
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_bnb, display_labels=['Regular', 'High Value'])
disp2.plot(ax=axes[0,1], cmap='Greens')
axes[0,1].set_title('Bernoulli NB - Confusion Matrix')

# ROC Curves
# GaussianNB ROC
fpr_gnb, tpr_gnb, _ = roc_curve(y_test, y_prob_gnb)
roc_auc_gnb = auc(fpr_gnb, tpr_gnb)
axes[1,0].plot(fpr_gnb, tpr_gnb, color='blue', lw=2, 
               label=f'Gaussian NB (AUC = {roc_auc_gnb:.3f})')
axes[1,0].plot([0, 1], [0, 1], color='red', lw=1, linestyle='--', label='Random')
axes[1,0].set_xlim([0.0, 1.0])
axes[1,0].set_ylim([0.0, 1.05])
axes[1,0].set_xlabel('False Positive Rate')
axes[1,0].set_ylabel('True Positive Rate')
axes[1,0].set_title('Gaussian NB - ROC Curve')
axes[1,0].legend(loc="lower right")
axes[1,0].grid(True, alpha=0.3)

# BernoulliNB ROC
fpr_bnb, tpr_bnb, _ = roc_curve(y_test_purchase, y_prob_bnb)
roc_auc_bnb = auc(fpr_bnb, tpr_bnb)
axes[1,1].plot(fpr_bnb, tpr_bnb, color='green', lw=2, 
               label=f'Bernoulli NB (AUC = {roc_auc_bnb:.3f})')
axes[1,1].plot([0, 1], [0, 1], color='red', lw=1, linestyle='--', label='Random')
axes[1,1].set_xlim([0.0, 1.0])
axes[1,1].set_ylim([0.0, 1.05])
axes[1,1].set_xlabel('False Positive Rate')
axes[1,1].set_ylabel('True Positive Rate')
axes[1,1].set_title('Bernoulli NB - ROC Curve')
axes[1,1].legend(loc="lower right")
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Model comparison
print(f"\n📊 MODEL COMPARISON - NAIVE BAYES")
print("=" * 50)
print(f"\n🔹 Gaussian Naive Bayes (RFM Features):")
print(f"   • ROC AUC: {roc_auc_gnb:.3f}")
print(f"   • Features: Recency, Frequency, Monetary")

print(f"\n🔹 Bernoulli Naive Bayes (Purchase Flags):")
print(f"   • ROC AUC: {roc_auc_bnb:.3f}")
print(f"   • Features: {X_purchase.shape[1]} binary purchase indicators")

better_model = "Gaussian NB" if roc_auc_gnb > roc_auc_bnb else "Bernoulli NB"
print(f"\n🏆 Better performing model: {better_model}")

print("\n✅ Naïve Bayes Classification Completed!")

## 6. Support Vector Machine (SVM)

We'll build SVM models for "High Value" customer prediction using:
- Linear SVM
- RBF (Radial Basis Function) SVM
- Grid search for hyperparameter optimization

In [123]:
# Import SVM libraries
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
import matplotlib.colors as mcolors

print("\n" + "="*50)
print("🧠 SUPPORT VECTOR MACHINE (SVM)")
print("="*50)

# Use the same train/test split as Naive Bayes for fair comparison
print(f"Using RFM features for SVM classification")
print(f"Training set: {X_train_rfm_scaled.shape}")
print(f"Test set: {X_test_rfm_scaled.shape}")

# Linear SVM
print(f"\n🔹 Training Linear SVM...")
svm_linear = SVC(kernel='linear', probability=True, random_state=42)
svm_linear.fit(X_train_rfm_scaled, y_train)

# Predictions
y_pred_linear = svm_linear.predict(X_test_rfm_scaled)
y_prob_linear = svm_linear.predict_proba(X_test_rfm_scaled)[:, 1]
accuracy_linear = accuracy_score(y_test, y_pred_linear)

print(f"Linear SVM Accuracy: {accuracy_linear:.3f}")
print(f"\nLinear SVM Classification Report:")
print(classification_report(y_test, y_pred_linear, target_names=['Regular', 'High Value']))

In [124]:
# RBF SVM with Grid Search
print(f"\n🔹 Training RBF SVM with Grid Search...")

# Define parameter grid
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1]
}

# Grid search with cross-validation
svm_rbf = SVC(kernel='rbf', probability=True, random_state=42)
grid_search = GridSearchCV(
    svm_rbf, 
    param_grid, 
    cv=5, 
    scoring='accuracy', 
    n_jobs=-1,
    verbose=1
)

# Fit grid search
grid_search.fit(X_train_rfm_scaled, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.3f}")

# Get best model
best_svm_rbf = grid_search.best_estimator_

# Predictions with best RBF SVM
y_pred_rbf = best_svm_rbf.predict(X_test_rfm_scaled)
y_prob_rbf = best_svm_rbf.predict_proba(X_test_rfm_scaled)[:, 1]
accuracy_rbf = accuracy_score(y_test, y_pred_rbf)

print(f"\nRBF SVM Accuracy: {accuracy_rbf:.3f}")
print(f"\nRBF SVM Classification Report:")
print(classification_report(y_test, y_pred_rbf, target_names=['Regular', 'High Value']))

In [125]:
# Visualize SVM decision boundaries using PCA
print(f"\n📊 Creating decision boundary visualizations...")

# Apply PCA to training and test data for visualization
pca_svm = PCA(n_components=2, random_state=42)
X_train_pca = pca_svm.fit_transform(X_train_rfm_scaled)
X_test_pca = pca_svm.transform(X_test_rfm_scaled)

# Train SVMs on PCA-reduced data for visualization
svm_linear_pca = SVC(kernel='linear', probability=True, random_state=42)
svm_rbf_pca = SVC(kernel='rbf', C=grid_search.best_params_['C'], 
                  gamma=grid_search.best_params_['gamma'], 
                  probability=True, random_state=42)

svm_linear_pca.fit(X_train_pca, y_train)
svm_rbf_pca.fit(X_train_pca, y_train)

# Create decision boundary plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Function to plot decision boundary
def plot_decision_boundary(ax, model, X, y, title):
    h = 0.02  # Step size in mesh
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                        np.arange(y_min, y_max, h))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdYlBu, edgecolors='black')
    ax.set_title(title)
    ax.set_xlabel(f'PC1 ({pca_svm.explained_variance_ratio_[0]:.2%} variance)')
    ax.set_ylabel(f'PC2 ({pca_svm.explained_variance_ratio_[1]:.2%} variance)')
    return scatter

# Plot Linear SVM
scatter1 = plot_decision_boundary(ax1, svm_linear_pca, X_test_pca, y_test, 
                                 'Linear SVM Decision Boundary')

# Plot RBF SVM
scatter2 = plot_decision_boundary(ax2, svm_rbf_pca, X_test_pca, y_test, 
                                 f'RBF SVM Decision Boundary\n(C={grid_search.best_params_["C"]}, γ={grid_search.best_params_["gamma"]})')

# Add colorbar
cbar = plt.colorbar(scatter1, ax=[ax1, ax2], orientation='horizontal', 
                   fraction=0.05, pad=0.1)
cbar.set_label('Customer Type (0=Regular, 1=High Value)')

plt.tight_layout()
plt.show()

In [126]:
# Create comprehensive model comparison
print("\n" + "="*60)
print("📊 COMPREHENSIVE MODEL COMPARISON")
print("="*60)

# Calculate ROC AUCs for SVM models
fpr_linear, tpr_linear, _ = roc_curve(y_test, y_prob_linear)
roc_auc_linear = auc(fpr_linear, tpr_linear)

fpr_rbf, tpr_rbf, _ = roc_curve(y_test, y_prob_rbf)
roc_auc_rbf = auc(fpr_rbf, tpr_rbf)

# Create summary table
model_results = pd.DataFrame({
    'Model': ['Gaussian NB', 'Bernoulli NB', 'Linear SVM', 'RBF SVM'],
    'Test Accuracy': [accuracy_score(y_test, y_pred_gnb), 
                     accuracy_score(y_test_purchase, y_pred_bnb),
                     accuracy_linear, 
                     accuracy_rbf],
    'ROC AUC': [roc_auc_gnb, roc_auc_bnb, roc_auc_linear, roc_auc_rbf],
    'Features': ['RFM (3)', f'Purchase Flags ({X_purchase.shape[1]})', 'RFM (3)', 'RFM (3)']
})

print("\n📈 MODEL PERFORMANCE SUMMARY:")
print("-" * 60)
print(model_results.to_string(index=False, float_format='%.3f'))

# Find best model
best_model_idx = model_results['ROC AUC'].idxmax()
best_model_name = model_results.loc[best_model_idx]['Model']
best_auc = model_results.loc[best_model_idx]['ROC AUC']

print(f"\n🏆 BEST PERFORMING MODEL: {best_model_name}")
print(f"   • ROC AUC: {best_auc:.3f}")
print(f"   • Test Accuracy: {model_results.loc[best_model_idx]['Test Accuracy']:.3f}")

# Plot ROC comparison
plt.figure(figsize=(10, 8))
plt.plot(fpr_gnb, tpr_gnb, label=f'Gaussian NB (AUC = {roc_auc_gnb:.3f})', linewidth=2)
plt.plot(fpr_bnb, tpr_bnb, label=f'Bernoulli NB (AUC = {roc_auc_bnb:.3f})', linewidth=2)
plt.plot(fpr_linear, tpr_linear, label=f'Linear SVM (AUC = {roc_auc_linear:.3f})', linewidth=2)
plt.plot(fpr_rbf, tpr_rbf, label=f'RBF SVM (AUC = {roc_auc_rbf:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Model Comparison')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

print("\n✅ Support Vector Machine Analysis Completed!")

## 7. Conclusions & Business Recommendations

### Key Findings Summary

Based on our comprehensive analysis of the UCI Online Retail dataset, here are the main insights and business recommendations:

In [127]:
# Final Analysis Summary
print("📊 SMART RETAIL ANALYTICS - FINAL INSIGHTS")
print("=" * 60)

print(f"\n📝 Dataset Overview:")
print(f"   • Total transactions analyzed: {len(df_clean):,}")
print(f"   • Unique customers: {len(rfm):,}")
print(f"   • Analysis period: {df_clean['InvoiceDate'].min().strftime('%Y-%m-%d')} to {df_clean['InvoiceDate'].max().strftime('%Y-%m-%d')}")
print(f"   • High-value customers (>75th percentile): {rfm_classification['HighValue'].sum()} ({rfm_classification['HighValue'].mean()*100:.1f}%)")

print(f"\n🔍 Customer Similarity Insights:")
print(f"   • Average RFM distance between customers: {non_diag_euclidean.mean():.3f}")
print(f"   • Average product overlap (Jaccard): {non_diag_jaccard.mean():.3f}")
print(f"   • Customer behavior shows moderate diversity in both RFM and product preferences")

print(f"\n🎯 Clustering Results:")
print(f"   • Optimal K-Means clusters: {optimal_k}")
print(f"   • DBSCAN identified {n_clusters_dbscan} clusters with {n_noise/len(dbscan_labels)*100:.1f}% noise points")
print(f"   • Clear customer segments identified for targeted marketing")

if len(rules_fpgrowth) > 0 and 'rules_fpgrowth_sorted' in locals():
    print(f"\n🛍️ Association Rules:")
    print(f"   • {len(rules_fpgrowth)} meaningful product associations discovered")
    print(f"   • Average lift: {rules_fpgrowth['lift'].mean():.2f}")
    # Create rule description for strongest association
    strongest_rule = rules_fpgrowth_sorted.iloc[0]
    strongest_rule_desc = f"{list(strongest_rule['antecedents'])[0]} → {list(strongest_rule['consequents'])[0]}"
    print(f"   • Strongest association: {strongest_rule_desc} (Lift: {strongest_rule['lift']:.2f})")
else:
    print(f"\n🛍️ Association Rules: Limited associations found - consider product bundling strategies")

print(f"\n🧠 Predictive Model Performance:")
print(f"   • Best model: {best_model_name} (AUC: {best_auc:.3f})")
print(f"   • Can reliably identify high-value customers for targeted campaigns")

print(f"\n💼 BUSINESS RECOMMENDATIONS:")
print(f"\n1. 🎯 Customer Segmentation Strategy:")
if optimal_k > 1:
    print(f"   • Implement {optimal_k}-tier customer segmentation based on RFM analysis")
    print(f"   • Develop targeted marketing campaigns for each segment")
    print(f"   • Focus retention efforts on high-value customers (top 25%)")

print(f"\n2. 🛍️ Product Strategy:")
if len(rules_fpgrowth) > 0:
    print(f"   • Implement cross-selling based on {len(rules_fpgrowth)} association rules")
    print(f"   • Create product bundles for frequently bought together items")
else:
    print(f"   • Investigate product catalog for better cross-selling opportunities")
print(f"   • Stock popular items identified in frequency analysis")

print(f"\n3. 📊 Predictive Analytics:")
print(f"   • Deploy {best_model_name} model for real-time customer value prediction")
print(f"   • Automate high-value customer identification with {best_auc:.1%} accuracy")
print(f"   • Use RFM features for ongoing customer health monitoring")

print(f"\n4. 🚀 Operational Improvements:")
print(f"   • Prioritize customer service for high-value segments")
print(f"   • Implement early warning system for at-risk valuable customers")
print(f"   • Optimize inventory based on customer segment preferences")

print(f"\n" + "=" * 60)
print(f"✅ ANALYSIS COMPLETE - Ready for Business Implementation!")
print(f"=" * 60)

---

### 🎆 **Project Completed Successfully!**

This comprehensive analysis demonstrates the power of data mining techniques in retail analytics. The insights generated can drive strategic business decisions, improve customer relationships, and optimize operational efficiency.

**Key Deliverables:**
1. ✅ Clean, preprocessed dataset with RFM features
2. ✅ Customer similarity analysis using Euclidean and Jaccard measures
3. ✅ Customer segmentation using K-Means and DBSCAN
4. ✅ Product association rules via Apriori and FP-Growth
5. ✅ High-value customer prediction models (Naïve Bayes & SVM)
6. ✅ Comprehensive visualizations and business recommendations
